## MLflow Tracking + Light Fine-Tuning of BLIP


## Required Libraries

In [4]:
%pip install -q mlflow


Note: you may need to restart the kernel to use updated packages.


In [5]:
import json
import random
import sys
from pathlib import Path

import mlflow
import pandas as pd
import torch
import transformers
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import BlipForConditionalGeneration, BlipProcessor

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("MLflow version:", mlflow.__version__)


Python version: 3.13.5
PyTorch version: 2.13.0+cpu
Transformers version: 5.13.0
MLflow version: 3.14.0


## Project Paths + MLflow Config

In [6]:
CAPTION_FILE = Path("../data/flickr8k/captions.txt")
IMAGE_DIR = Path("../data/flickr8k/Images")

assert CAPTION_FILE.exists(), f"CAPTION_FILE not found: {CAPTION_FILE.resolve()}"
assert IMAGE_DIR.exists(), f"IMAGE_DIR not found: {IMAGE_DIR.resolve()}"

MODEL_NAME = "Salesforce/blip-image-captioning-base"

DAY2_OUTPUT_DIR = Path("outputs/day2_baseline")
DAY3_OUTPUT_DIR = Path("outputs/day3_finetune")
DAY3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"  # file-store backend is deprecated in MLflow 3.x
MLFLOW_EXPERIMENT_NAME = "BLIP-Flickr8k-Captioning"

RANDOM_SEED = 42

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", MLFLOW_EXPERIMENT_NAME)


2026/07/16 16:46:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/16 16:46:36 INFO mlflow.store.db.utils: Updating database tables
2026/07/16 16:46:41 INFO mlflow.tracking.fluent: Experiment with name 'BLIP-Flickr8k-Captioning' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db
Experiment: BLIP-Flickr8k-Captioning


## Step 1 — Log the Baseline Run to MLflow

In [7]:
with open(DAY2_OUTPUT_DIR / "day2_run_config.json") as f:
    day2_config = json.load(f)

day2_config


{'model_name': 'Salesforce/blip-image-captioning-base',
 'sample_size': 300,
 'random_seed': 42,
 'generation_config': {'max_new_tokens': 30,
  'num_beams': 5,
  'early_stopping': True},
 'metrics': {'BLEU': 20.37,
  'ROUGE-1': 50.73,
  'ROUGE-2': 27.21,
  'ROUGE-L': 47.77,
  'METEOR': 36.88}}

In [8]:
with mlflow.start_run(run_name="zero_shot_blip_baseline") as run:
    mlflow.set_tag("stage", "day2_baseline")
    mlflow.set_tag("fine_tuned", "false")

    mlflow.log_param("model_name", day2_config["model_name"])
    mlflow.log_param("sample_size", day2_config["sample_size"])
    mlflow.log_param("random_seed", day2_config["random_seed"])
    for key, value in day2_config["generation_config"].items():
        mlflow.log_param(f"gen_{key}", value)

    for metric_name, metric_value in day2_config["metrics"].items():
        if isinstance(metric_value, (int, float)):
            safe_name = metric_name.lower().replace(" ", "_").replace("-", "_")
            mlflow.log_metric(safe_name, metric_value)

    mlflow.log_artifact(str(DAY2_OUTPUT_DIR / "day2_per_image_results.csv"))
    mlflow.log_artifact(str(DAY2_OUTPUT_DIR / "day2_baseline_metrics_summary.csv"))

    baseline_run_id = run.info.run_id
    print("Logged baseline run:", baseline_run_id)


Logged baseline run: e72f1446dc154123bc19b98c38b981af


## Step 2 — Light Fine-Tuning Setup
Freeze the **vision encoder**; fine-tune only the **text decoder** on a small Flickr8k training subset, disjoint from Day 2's validation subset.

In [9]:
TRAIN_SAMPLE_SIZE = 80      # small, CPU-friendly training subset
NUM_EPOCHS = 2
BATCH_SIZE = 2
LEARNING_RATE = 5e-5

captions = pd.read_csv(CAPTION_FILE)
captions.columns = [
    str(column).strip().lower().replace(" ", "_") for column in captions.columns
]

day2_results = pd.read_csv(DAY2_OUTPUT_DIR / "day2_per_image_results.csv")
validation_images = set(day2_results["image"])

random.seed(RANDOM_SEED)
available_images = [
    image_name for image_name in captions["image"].drop_duplicates().tolist()
    if image_name not in validation_images and (IMAGE_DIR / image_name).exists()
]

if len(available_images) < TRAIN_SAMPLE_SIZE:
    print(f"Warning: only {len(available_images)} training images available; reducing sample.")
    TRAIN_SAMPLE_SIZE = len(available_images)

train_images = random.sample(available_images, TRAIN_SAMPLE_SIZE)
print(f"Training subset: {len(train_images)} images (disjoint from Day 2 validation subset).")


Training subset: 80 images (disjoint from Day 2 validation subset).


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

# Freeze the vision encoder; fine-tune only the text decoder.
for parameter in model.vision_model.parameters():
    parameter.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} "
      f"({100 * trainable_params / total_params:.1f}%)")


Using device: cpu


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 3397.14it/s]


Trainable parameters: 137,881,148 / 223,971,644 (61.6%)


In [11]:
class Flickr8kCaptionDataset(Dataset):
    """One (image, single caption) pair per item; one caption per image per epoch is
    sampled randomly for light fine-tuning simplicity."""

    def __init__(self, image_names: list[str], captions_df: pd.DataFrame, image_dir: Path):
        self.image_names = image_names
        self.captions_df = captions_df
        self.image_dir = image_dir

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, index):
        image_name = self.image_names[index]
        image_path = self.image_dir / image_name

        with Image.open(image_path) as image:
            rgb_image = image.convert("RGB")

        candidate_captions = (
            self.captions_df.loc[self.captions_df["image"] == image_name, "caption"]
            .astype(str)
            .tolist()
        )
        target_caption = random.choice(candidate_captions)

        return rgb_image, target_caption


def collate_fn(batch):
    images, target_captions = zip(*batch)

    inputs = processor(
        images=list(images),
        text=list(target_captions),
        return_tensors="pt",
        padding=True,
        truncation=True,
    )
    return inputs


train_dataset = Flickr8kCaptionDataset(train_images, captions, IMAGE_DIR)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)

print(f"Training batches per epoch: {len(train_loader)}")


Training batches per epoch: 40


## Step 3 — Fine-Tune the Text Decoder (Logged to MLflow)

In [12]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE
)

with mlflow.start_run(run_name="blip_text_decoder_finetune") as run:
    mlflow.set_tag("stage", "day3_finetune")
    mlflow.set_tag("fine_tuned", "true")
    mlflow.set_tag("frozen_component", "vision_encoder")

    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("train_sample_size", len(train_images))
    mlflow.log_param("num_epochs", NUM_EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("trainable_parameters", trainable_params)
    mlflow.log_param("random_seed", RANDOM_SEED)
    mlflow.log_param("device", str(device))

    model.train()
    global_step = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_loss_total = 0.0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")
        for batch in progress_bar:
            batch = {name: tensor.to(device) for name, tensor in batch.items()}

            outputs = model(
                input_ids=batch["input_ids"],
                pixel_values=batch["pixel_values"],
                attention_mask=batch["attention_mask"],
                labels=batch["input_ids"],
            )
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss_total += loss.item()
            global_step += 1
            mlflow.log_metric("train_loss_step", loss.item(), step=global_step)
            progress_bar.set_postfix(loss=loss.item())

        epoch_avg_loss = epoch_loss_total / len(train_loader)
        mlflow.log_metric("train_loss_epoch", epoch_avg_loss, step=epoch)
        print(f"Epoch {epoch}: average training loss = {epoch_avg_loss:.4f}")

    model.eval()

    checkpoint_dir = DAY3_OUTPUT_DIR / "blip_finetuned_decoder"
    model.save_pretrained(checkpoint_dir)
    processor.save_pretrained(checkpoint_dir)
    mlflow.log_artifacts(str(checkpoint_dir), artifact_path="model_checkpoint")

    finetune_run_id = run.info.run_id
    print("Logged fine-tuning run:", finetune_run_id)


Epoch 1/2: 100%|██████████| 40/40 [06:22<00:00,  9.56s/it, loss=4.06]


Epoch 1: average training loss = 3.9464


Epoch 2/2: 100%|██████████| 40/40 [05:10<00:00,  7.76s/it, loss=4.04]


Epoch 2: average training loss = 3.0032


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.44s/it]


Logged fine-tuning run: ef83d5cc039344bc98498abd1a08d653


## Step 4 — View Runs in the MLflow UI

In [13]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
runs = client.search_runs(experiment_ids=[experiment.experiment_id])

for run in runs:
    print(run.info.run_name, "-", run.info.run_id, "-", run.info.status)


blip_text_decoder_finetune - ef83d5cc039344bc98498abd1a08d653 - FINISHED
zero_shot_blip_baseline - e72f1446dc154123bc19b98c38b981af - FINISHED
